# Top-2 Models — live transcription demo  ·  run on Kaggle (GPU)

Attach the demo samples (the two `Fleurs_*_Samples` folders), then: **choose dataset → load model → pick index → run.**

In [4]:
!pip install -q jiwer hazm sacrebleu soundfile pyroomacoustics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 33.3 MB/s eta 0:00:0000:01


In [5]:
# ── BUILD DATASETS into /kaggle/working (run once per session; ~15–40 min) ──
import os, io, csv, tarfile, subprocess, tempfile
import numpy as np, soundfile as sf, requests
from scipy.signal import butter, sosfilt

REPO, LANG, SPLIT = "google/fleurs", "fa_ir", "test"
N_CLEAN, N_NOISY_BASE = None, 200        # None = all clean (~871); 200×5 = 1000 noisy
SNR_DB, PROFILE_REF_SNR = [20, 15, 10, 5, 0], 10
OUT_CLEAN = "/kaggle/working/Fleurs_Clean_Samples"
OUT_NOISY = "/kaggle/working/Fleurs_Noisy_Synth_Samples"
BASE = f"https://huggingface.co/datasets/{REPO}/resolve/main/data/{LANG}"
TSV_URL, TAR_URL = f"{BASE}/{SPLIT}.tsv", f"{BASE}/audio/{SPLIT}.tar.gz"
_tok = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN")
HEADERS = {"Accept-Encoding": "identity"}
if _tok: HEADERS["Authorization"] = f"Bearer {_tok}"

def _rms(x): return float(np.sqrt(np.mean(np.asarray(x, np.float64) ** 2)) + 1e-12)
def _fit(x, n):
    if len(x) == 0: return np.zeros(n, np.float32)
    if len(x) < n: x = np.tile(x, int(np.ceil(n / len(x))))
    return x[:n].astype(np.float32)
def reverb_synth(a, sr, rng, rt60=0.4, **k):
    n = max(1, int(sr * rt60)); t = np.arange(n)
    ir = (rng.standard_normal(n) * np.exp(-6.908 * t / (rt60 * sr))).astype(np.float32); ir[0] += 1.0
    o = np.convolve(a, ir)[:len(a)]; return (o * (_rms(a) / _rms(o))).astype(np.float32)
def reverb_pyroom(a, sr, rng, rt60=0.4, room=(4.0, 5.0, 3.0), **k):
    try:
        import pyroomacoustics as pra
        e, mo = pra.inverse_sabine(rt60, list(room))
        r = pra.ShoeBox(list(room), fs=sr, materials=pra.Material(e), max_order=int(mo))
        r.add_source([room[0]*0.5, room[1]*0.35, 1.2], signal=a.astype(np.float64))
        r.add_microphone(np.array([room[0]*0.5, room[1]*0.65, 1.2]).reshape(3, 1)); r.simulate()
        o = _fit(np.asarray(r.mic_array.signals[0], np.float32)[:len(a)], len(a))
        return (o * (_rms(a) / _rms(o))).astype(np.float32)
    except Exception as ex:
        print("   reverb fallback:", ex); return reverb_synth(a, sr, rng, rt60=rt60)
def add_gaussian(a, sr, rng, snr_db=10, **k):
    p = float(np.mean(a.astype(np.float64) ** 2)) + 1e-12
    return (a + rng.normal(0, np.sqrt(p / (10 ** (snr_db / 10))), size=a.shape).astype(np.float32)).astype(np.float32)
def bandlimit(a, sr, rng, low=120.0, high=6000.0, order=4, **k):
    high = min(high, sr / 2 - 1)
    return sosfilt(butter(order, [low, high], btype="band", fs=sr, output="sos"), a).astype(np.float32)
def clip_dist(a, sr, rng, drive=0.2, **k):
    return np.clip(a * (1.0 + drive * 6.0), -1.0, 1.0).astype(np.float32)
def codec_roundtrip(a, sr, rng, codec="opus", bitrate="20k", **k):
    try:
        with tempfile.TemporaryDirectory() as d:
            wi, ec, wo = (os.path.join(d, f) for f in ("i.wav", "e.opus", "o.wav"))
            sf.write(wi, np.clip(a, -1, 1).astype(np.float32), sr, subtype="PCM_16")
            subprocess.run(["ffmpeg", "-y", "-i", wi, "-c:a", "libopus", "-b:a", bitrate, ec], check=True, capture_output=True)
            subprocess.run(["ffmpeg", "-y", "-i", ec, "-ar", str(sr), "-ac", "1", wo], check=True, capture_output=True)
            y, _ = sf.read(wo, dtype="float32")
        return _fit(np.asarray(y, np.float32), len(a))
    except Exception as ex:
        print("   codec skipped:", ex); return a
_EFF = {"reverb_pyroom": (reverb_pyroom, "reverb"), "add_gaussian": (add_gaussian, "noise"),
        "bandlimit": (bandlimit, "mic"), "clip_dist": (clip_dist, "mic"), "codec_roundtrip": (codec_roundtrip, "mic")}
PIPE = [("reverb_pyroom", dict(rt60=0.45, room=(4.0, 5.0, 3.0))), ("add_gaussian", dict(snr_db=PROFILE_REF_SNR)),
        ("bandlimit", dict(low=120.0, high=6000.0)), ("clip_dist", dict(drive=0.15)),
        ("codec_roundtrip", dict(codec="opus", bitrate="20k"))]
def apply_profile(a, sr, snr, rng):
    x = np.asarray(a, np.float32).copy(); off = snr - PROFILE_REF_SNR
    for nm, pp in PIPE:
        fn, cat = _EFF[nm]; p = dict(pp)
        if cat == "noise": p["snr_db"] = p.get("snr_db", PROFILE_REF_SNR) + off
        x = fn(x, sr, rng, **p)
    return x.astype(np.float32)
def load_transcripts():
    r = requests.get(TSV_URL, headers=HEADERS, timeout=120); r.raise_for_status(); refs = {}
    for ln in r.text.splitlines():
        c = ln.split("\t")
        if len(c) >= 3: refs[c[1].strip()] = (c[3] if len(c) > 3 and c[3].strip() else c[2]).strip()
    return refs
def fetch_base_clips(n=None):
    refs, clips = load_transcripts(), []
    with requests.get(TAR_URL, headers=HEADERS, stream=True, timeout=600) as resp:
        resp.raise_for_status(); resp.raw.decode_content = False
        with tarfile.open(fileobj=resp.raw, mode="r|gz") as tar:
            for m in tar:
                if not (m.isfile() and m.name.endswith(".wav")): continue
                fn = os.path.basename(m.name)
                arr, sr = sf.read(io.BytesIO(tar.extractfile(m).read()), dtype="float32")
                if arr.ndim > 1: arr = arr.mean(axis=1)
                clips.append({"file": fn, "array": arr.astype(np.float32), "sr": sr, "reference": refs.get(fn, "")})
                if n is not None and len(clips) >= n: break
    return clips

if os.path.exists(os.path.join(OUT_CLEAN, "metadata.csv")) and os.path.exists(os.path.join(OUT_NOISY, "metadata.csv")):
    print("datasets already built this session — skipping.")
else:
    os.makedirs(OUT_CLEAN, exist_ok=True); os.makedirs(OUT_NOISY, exist_ok=True)
    print("downloading FLEURS (~844 MB) ...")
    base = fetch_base_clips(N_CLEAN)
    print(f"fetched {len(base)} clean clips; writing ...")
    with open(os.path.join(OUT_CLEAN, "metadata.csv"), "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=["index", "filename", "reference"]); w.writeheader()
        for i, b in enumerate(base):
            fn = f"clip_{i:03d}.wav"; sf.write(os.path.join(OUT_CLEAN, fn), b["array"], b["sr"], subtype="PCM_16")
            w.writerow({"index": i, "filename": fn, "reference": b["reference"]})
    print(f"clean: {len(base)} -> {OUT_CLEAN}")
    rng = np.random.default_rng(1234); n = 0
    with open(os.path.join(OUT_NOISY, "metadata.csv"), "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=["index", "filename", "source_index", "snr_db", "reference"]); w.writeheader()
        for si in range(min(N_NOISY_BASE, len(base))):
            b = base[si]
            for snr in SNR_DB:
                noisy = apply_profile(b["array"], b["sr"], snr, rng)
                fn = f"clip_{n:04d}_snr{snr}.wav"
                sf.write(os.path.join(OUT_NOISY, fn), np.clip(noisy, -1, 1), b["sr"], subtype="PCM_16")
                w.writerow({"index": n, "filename": fn, "source_index": si, "snr_db": snr, "reference": b["reference"]})
                n += 1
            if (si + 1) % 25 == 0: print(f"  noisy base {si+1}/{N_NOISY_BASE}")
    print(f"noisy: {n} -> {OUT_NOISY}")
print("build done — now run the setup cell")

downloading FLEURS (~844 MB) ...
fetched 871 clean clips; writing ...
clean: 871 -> /kaggle/working/Fleurs_Clean_Samples
  noisy base 25/200
  noisy base 50/200
  noisy base 75/200
  noisy base 100/200
  noisy base 125/200
  noisy base 150/200
  noisy base 175/200
  noisy base 200/200
noisy: 1000 -> /kaggle/working/Fleurs_Noisy_Synth_Samples
build done — now run the setup cell


In [6]:
# ── setup: utilities, metrics, sample loader, model loader, display ──
import os, csv, time, gc
import numpy as np, soundfile as sf, torch
from transformers import (WhisperForConditionalGeneration, WhisperProcessor,
                          SeamlessM4Tv2Model, AutoProcessor)
from jiwer import wer, cer, process_words
from sacrebleu.metrics import CHRF
from collections import Counter
import hazm
from IPython.display import Audio, HTML, display

device = "cuda" if torch.cuda.is_available() else "cpu"
_norm, _chrf = hazm.Normalizer(), CHRF()
import re
def normalize_persian(t):
    t = _norm.normalize(t or ""); t = re.sub(r"[^\w\s]", "", t, flags=re.UNICODE)
    return re.sub(r"\s+", " ", t).strip()
def _script_contam(t):
    a = [c for c in t if c.isalpha()]
    if not a: return 0.0
    rng = [('؀','ۿ'),('ﭐ','﷿'),('ﹰ','﻿')]
    return sum(1 for c in a if not any(lo<=c<=hi for lo,hi in rng))/len(a)
def _rep(t, n=4):
    w = t.split()
    if len(w) < n+1: return 0.0
    g = [tuple(w[i:i+n]) for i in range(len(w)-n+1)]
    return sum(v-1 for v in Counter(g).values() if v>1)/len(g)

def compute_metrics(ref, hyp):
    rn, hn = normalize_persian(ref), normalize_persian(hyp)
    try: w, c = wer(rn, hn), cer(rn, hn)
    except Exception: w = c = float("nan")
    try: ch = _chrf.sentence_score(hn, [rn]).score
    except Exception: ch = float("nan")
    try:
        pw = process_words([rn], [hn]); tot = pw.hits + pw.substitutions + pw.deletions
        sub, ins, dl = pw.substitutions/tot, pw.insertions/tot, pw.deletions/tot
    except Exception: sub = ins = dl = float("nan")
    rw, hw = len(rn.split()), len(hn.split()); hr = hw/max(rw, 1)
    contam, rep = _script_contam(hyp), _rep(hyp)
    halluc = int(rep > 0.2 or hr > 2.0 or contam > 0.5)
    return {"WER": round(w,3), "CER": round(c,3), "chrF": round(ch,1),
            "Sub": round(sub,3), "Ins": round(ins,3), "Del": round(dl,3),
            "halluc_ratio": round(hr,2), "is_hallucination": halluc}

# samples
def _find_dir(name):
    for base in ["/kaggle/input", "/kaggle/working", "."]:
        if not os.path.isdir(base): continue
        for dp, _, fns in os.walk(base):
            if os.path.basename(dp) == name and "metadata.csv" in fns: return dp
    return None
SAMPLE_DIRS = {"clean": _find_dir("Fleurs_Clean_Samples"), "noisy": _find_dir("Fleurs_Noisy_Synth_Samples")}
print("clean:", SAMPLE_DIRS["clean"], "\nnoisy:", SAMPLE_DIRS["noisy"])

def load_clips(dataset):
    folder = SAMPLE_DIRS[dataset]
    assert folder, f"samples for '{dataset}' not found — attach the Fleurs_*_Samples folder"
    clips = []
    for r in csv.DictReader(open(os.path.join(folder, "metadata.csv"), encoding="utf-8")):
        a, sr = sf.read(os.path.join(folder, r["filename"]), dtype="float32")
        if a.ndim > 1: a = a.mean(axis=1)
        clips.append({"index": int(r.get("index", len(clips))), "filename": r["filename"],
                      "reference": r.get("reference", ""), "snr_db": r.get("snr_db", ""),
                      "audio": a.astype(np.float32), "sr": int(sr)})
    clips.sort(key=lambda c: c["index"])
    print(f"[{dataset}] {len(clips)} clips loaded (index 0..{len(clips)-1})")
    return clips

# models (the top two)
TOP2 = {"seamless": "facebook/seamless-m4t-v2-large", "whisper": "nezamisafa/whisper-persian-v4"}
_CUR = {"obj": None}
def _to16k(a, sr):
    if sr == 16000: return a
    import torchaudio
    return torchaudio.functional.resample(torch.from_numpy(np.asarray(a, np.float32)).unsqueeze(0), sr, 16000).squeeze(0).numpy()
def load_model(key):
    old = _CUR.get("obj")
    if old: old["_m"] = old["_p"] = None; _CUR["obj"] = None
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    mid = TOP2[key]; print(f"Loading {mid} ...")
    if key == "seamless":
        p = AutoProcessor.from_pretrained(mid)
        mdl = SeamlessM4Tv2Model.from_pretrained(mid).to(device).eval()
        def fn(a, sr):
            a = _to16k(a, sr); inp = p(audio=a, sampling_rate=16000, return_tensors="pt")
            inp = {k: v.to(device) for k, v in inp.items()}
            with torch.no_grad(): out = mdl.generate(**inp, tgt_lang="pes", generate_speech=False)
            seqs = out.sequences if hasattr(out, "sequences") else out
            return p.tokenizer.batch_decode(seqs, skip_special_tokens=True)[0]
    else:
        dt = torch.float16 if device == "cuda" else torch.float32
        p = WhisperProcessor.from_pretrained(mid)
        mdl = WhisperForConditionalGeneration.from_pretrained(mid, torch_dtype=dt).to(device).eval()
        mdl.generation_config.forced_decoder_ids = None
        def fn(a, sr):
            a = _to16k(a, sr); feat = p(a, sampling_rate=16000, return_tensors="pt").input_features.to(device)
            if device == "cuda": feat = feat.half()
            with torch.no_grad(): ids = mdl.generate(feat)
            return p.batch_decode(ids, skip_special_tokens=True)[0]
    obj = {"name": mid, "fn": fn, "_m": mdl, "_p": p}; _CUR["obj"] = obj; print("Ready."); return obj

# display
def transcribe_demo(clips, model, index):
    clip = clips[index]; a, sr = clip["audio"], clip["sr"]
    t0 = time.perf_counter(); hyp = model["fn"](a, sr); dt = time.perf_counter() - t0
    mt = compute_metrics(clip["reference"], hyp)
    display(Audio(a, rate=sr))
    snr = f" · SNR {clip['snr_db']} dB" if str(clip.get("snr_db", "")) != "" else ""
    rows = "".join(f"<td style='padding:3px 10px;text-align:center'><b>{k}</b><br>{v}</td>" for k, v in mt.items())
    display(HTML(f"""<div style='font-family:Tahoma'>
      <div style='margin:4px 0'><b>Dataset:</b> {DATASET} &nbsp;|&nbsp; <b>Model:</b> {model['name']} &nbsp;|&nbsp;
        <b>Clip #{index}:</b> {clip['filename']}{snr} &nbsp;|&nbsp; {dt:.2f}s</div>
      <div dir='rtl' style='background:#eaf0ff;padding:8px;margin:5px 0;border-radius:5px'><b>مرجع:</b> {clip['reference']}</div>
      <div dir='rtl' style='background:#eaffea;padding:8px;margin:5px 0;border-radius:5px'><b>خروجی مدل:</b> {hyp}</div>
      <table style='border-collapse:collapse;margin-top:4px'><tr>{rows}</tr></table></div>"""))
print("setup ready")

clean: /kaggle/working/Fleurs_Clean_Samples 
noisy: /kaggle/working/Fleurs_Noisy_Synth_Samples
setup ready


## 1 · dataset

In [7]:
DATASET = "clean"        # "clean" | "noisy"
clips = load_clips(DATASET)

[clean] 871 clips loaded (index 0..870)


## 2 · model

In [8]:
MODEL = "seamless"       # "seamless" | "whisper"
model = load_model(MODEL)

Loading facebook/seamless-m4t-v2-large ...


preprocessor_config.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/5.17M [00:00<?, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Instantiating a decoder SeamlessM4Tv2Attention without passing `layer_idx` is not recommended and will lead to errors during the forward call, if caching is used. Please make sure to provide a `layer_idx` when creating this class.


Loading weights:   0%|          | 0/2232 [00:00<?, ?it/s]

generation_config.json: 0.00B [00:00, ?B/s]

Ready.


## 3 · pick index → transcribe

In [12]:
INDEX = 10
transcribe_demo(clips, model, INDEX)

WER0.0,CER0.0,chrF100.0,Sub0.0,Ins0.0,Del0.0,halluc_ratio1.0,is_hallucination0
